In [15]:
import sqlite3
from duckduckgo_search import DDGS

create the data base for DB tool 

In [16]:
con = sqlite3.connect("employee.db")
 
cursor=con.cursor()

cursor.execute(""" CREATE TABLE IF NOT EXISTS employee( 
              employee_id INTEGER,
              employee_name TEXT,
              manager TEXT
              )
            """)
cursor.execute("DELETE FROM employee")
cursor.execute(""" INSERT INTO employee VALUES (01, 'vinnu' ,'hari')
               """)

cursor.execute(""" INSERT INTO employee VALUES (02, 'yuri' ,'aries')
               """)

con.commit()
con.close()
print("database ready")

database ready


#database tool

In [17]:
def db_query(employee_id):
    con = sqlite3.connect("employee.db")
    cursor= con.cursor()
    cursor.execute("""
                SELECT employee_name, manager 
                FROM employee WHERE employee_id=? """, (int(employee_id),)
                )
    result =  cursor.fetchone()

    con.close()

    if result:
        return  (
            f"Employee:{result[0]},"
            f"Manager:{result[1]}"
        )
    return "Employee not found"

In [18]:
from ddgs import DDGS

In [19]:
def web_search(query):
    with DDGS() as ddgs:
        result =list(ddgs.text(query, max_results=5))

    print(result)
    
    if len(result)==0:
        return "NO result found"
    return result[0]["body"]
    # print(result)

In [46]:
web_search("who won ipl 2026?")

[{'title': 'Indian Premier League - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Indian_Premier_League', 'body': 'The current champions are the Royal Challengers Bengaluru, who won the 2026 season after defeating the Gujarat Titans in the final. ... 2026 IPL final to secure ...'}, {'title': 'Congratulations @royalchallengers.bengaluru winning the IPL 2026', 'href': 'https://www.instagram.com/p/DZAyLEESJB6/', 'body': 'May 31, 2026 ... For the second time, the Royal Challengers Bangalore are the champions of TATA IPL 2026. OCR. TATA IPL FINAL FOR THE SECOND TIME CHAMPIONS TATA ...'}, {'title': 'IPL 2026 Match Results | Full Scorecard & Summaries | IPLT20', 'href': 'https://www.iplt20.com/matches/results/2023', 'body': 'Final · MAY, MON 29 , 7:30 pm IST. Chennai Super Kings Won by 5 Wickets (D/L Method) (Winners) ; Qualifier 2 · MAY, FRI 26 , 7:30 pm IST. Gujarat Titans Won by 62 ...'}, {'title': 'RCB vs GT, IPL 2026 Final: Post Match Celebrations - YouTube', 'href': 'https://www.yo

'The current champions are the Royal Challengers Bengaluru, who won the 2026 season after defeating the Gujarat Titans in the final. ... 2026 IPL final to secure ...'

In [21]:
web_search("linkedin profile of microsoft ceo?")

[{'title': 'Satya Nadella - Chairman and CEO at Microsoft | LinkedIn', 'href': 'https://www.linkedin.com/in/satyanadella/', 'body': '2 weeks ago - Chairman and CEO at Microsoft · As chairman and CEO of Microsoft, I define my mission and that of my company as empowering every person and every organization on the planet to achieve more.'}, {'title': 'Ryan Roslansky - Executive Vice President at Microsoft | LinkedIn', 'href': 'https://www.linkedin.com/in/ryanroslansky/', 'body': '3 weeks ago - Ryan also serves on the Board of Trustees for the Paley Center for Media · Experience: Microsoft · Location: San Francisco Bay Area · 500+ connections on LinkedIn. View Ryan Roslansky’s profile on LinkedIn, ...'}, {'title': 'Brad Smith - Vice Chair and President at Microsoft Corporation | LinkedIn', 'href': 'https://www.linkedin.com/in/bradsmi/', 'body': "July 9, 2025 - Vice Chair and President at Microsoft Corporation · Serve as Microsoft's vice chair and president, leading work on a wide range of 

'2 weeks ago - Chairman and CEO at Microsoft · As chairman and CEO of Microsoft, I define my mission and that of my company as empowering every person and every organization on the planet to achieve more.'

graceful failure and retry logic

In [22]:
import time
def retry(func):
    def grace(*args):
        retry=3

        for attempt in range(retry):
            try:
                result=func(*args)
                return result
            except Exception as e:
                print(f"Retry {attempt +1}")
                time.sleep(1)
        
        return "Tool Failed"
    return grace

In [23]:
db_query_retry = retry(db_query)
web_search_retry=retry(web_search)

In [24]:
from langchain.tools import tool

In [25]:
@tool
def db_query_tool(employee_id: int)-> str:
    """ retrieve Employee information from database"""
    return db_query_retry(employee_id)

In [26]:
@tool
def web_search_tool(query:str)->str:
    """ search in the internet for inforamtion"""
    return web_search_retry(query)

In [27]:
print(web_search_tool.invoke("Sam Altman current company?"))

[{'title': 'Sam Altman - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Sam_Altman', 'body': '3 days ago - Samuel Harris Altman (born April 22, 1985) is an American entrepreneur and investor who has been the chief executive officer (CEO) of the artificial intelligence company OpenAI since 2019. Altman attended Stanford University for two years before he dropped out and co-founded Loopt, a smartphone ...'}, {'title': 'Sam Altman | Biography, OpenAI, ChatGPT, & Microsoft | Britannica Money', 'href': 'https://www.britannica.com/money/Sam-Altman', 'body': '2 weeks ago - Sam Altman is an American entrepreneur who was president of the start-up accelerator Y Combinator from 2014 to 2019 and chief executive officer of the artificial intelligence company OpenAI since 2019.'}, {'title': "Sam Altman : As ChatGPT maker OpenAI files for IPO, CEO Sam Altman shares 'current plan' for the company | - The Times of India", 'href': 'https://timesofindia.indiatimes.com/technology/tech-news/as-chatgpt-

In [28]:
import boto3
import json
import os
from dotenv import load_dotenv


# AWS Credentials
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")
load_dotenv("myenv.env")

# bedrock_runtime = boto3.client(
#     service_name="bedrock-runtime",
#     region_name=AWS_REGION,
#     aws_access_key_id=AWS_ACCESS_KEY_ID,
#     aws_secret_access_key=AWS_SECRET_ACCESS_KEY
# )

# MODEL_ID = "amazon.nova-micro-v1:0"

True

In [29]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="amazon.nova-micro-v1:0",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

In [30]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[
        db_query_tool,
        web_search_tool
    ],
    system_prompt="""
    You are a ReAct agent.
    Use tools which are required.
    Think step-by-step before answering.
    """
)

In [31]:
def Agent(question):

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        }
    )

    return response["messages"][-1].content

In [32]:
print(Agent("who manages employee 01 and work as manager?"))

<thinking> Now that I have retrieved the information from the database, I can see that the manager of employee 01 (vinnu) is hari. To confirm this information, I could search for hari's role and responsibilities, but it seems like the database already provided the necessary information to answer the question.</thinking> 

The manager of employee 01 and who works as a manager is Hari.


LangGraph is a framework to build AI agents as a flow of connected steps (nodes).

Without LangGraph:

question → LLM → answer

Problems:

❌ No memory

❌ No workflow

❌ No decision making

❌ Hard to control

With LangGraph:

Question
     ↓
Router
     ↓
Choose Tool
     ↓
Execute
     ↓
Answer

It behaves like an agent.

Main Components of LangGraph

There are only few concepts.

1. State:Carries information
2. Nodes:= One step in workflow.
3. Edges:connection between nodes.
4. Conditional Edges:Path depends on condition.
5.compile

routernode:This is the brain.

It decides:

Should I use:

Database?
Web Search?
answer node:Answer Node

This is final response generator.

It takes:

Question
+
Tool Result

and creates final answer.


User Question
      ↓
LLM Agent
      ↓
Decides Tool
     /      \
DB Query   Web Search
     \      /
      Final Answer

Create State

In [34]:
from typing import TypedDict

class AgentState(TypedDict):
    question: str
    tool: str
    result: str
    answer: str

Router Node

In [49]:
def router(state):
    question = state["question"].lower()

    db_keywords = [
        "employee",
        "manager",
        "department",
        "salary",
        "reports",
        "reporting"
    ]

    if any(word in question for word in db_keywords):
        state["tool"] = "db"
    else:
        state["tool"] = "web"

    return state

DB Node

In [50]:
def db_node(state: AgentState):

    question = state["question"]

    employee_id = "".join(
        filter(str.isdigit, question)
    )

    result = db_query_retry(employee_id)

    state["result"] = str(result)

    return state

Web Node

In [51]:
def web_node(state: AgentState):

    result = web_search_retry(
        state["question"]
    )

    state["result"] = result

    return state

Final Answer Node

In [52]:
def answer_node(state: AgentState):

    state["answer"] = (
        f"Answer: {state['result']}"
    )

    return state

Conditional Function

In [53]:
def choose_tool(state: AgentState):

    return state["tool"]

Build Graph

In [55]:
from langgraph.graph import (
    StateGraph,
    END
)

graph = StateGraph(AgentState)

graph.add_node("router", router)
graph.add_node("db", db_node)
graph.add_node("web", web_node)
graph.add_node("answer", answer_node)

graph.set_entry_point("router")

Conditional Branching

In [56]:
graph.add_conditional_edges(
    "router",
    choose_tool,
    {
        "db": "db",
        "web": "web"
    }
)

Remaining Edges

In [57]:
graph.add_edge("db", "answer")
graph.add_edge("web", "answer")
graph.add_edge("answer", END)

In [58]:
app = graph.compile()

In [60]:
result = app.invoke(
    {
        "question":
        "who is manager of employee 01?"
        #"Who reports to manager Hari?"
    }
)

print(result["answer"])

Answer: Employee:vinnu,Manager:hari


In [64]:
result = app.invoke(
    {
        "question":
        "how is the manager of employee_id 03?"
    }
)

print(result["answer"])

Answer: Employee not found


                 User Question
                        |
                        v
                    Router Node
                   /            \
                  /              \
            employee?            other?
              /                     \
             v                       v
         DB Node                Web Node
             \                     /
              \                   /
                   Answer Node
                        |
                       END